In [24]:
! uv pip install langchain-google-genai google-generativeai

Using Python 3.12.13 environment at: c:\Users\palla\Desktop\LLMOPS\.venv
Resolved 54 packages in 4.83s
 Downloaded google-ai-generativelanguage
 Downloaded google-api-python-client
Prepared 11 packages in 7.51s
Uninstalled 5 packages in 715ms
Installed 13 packages in 497ms
 - google-ai-generativelanguage==0.6.18
 + google-ai-generativelanguage==0.6.15
 + google-api-python-client==2.198.0
 + google-auth-httplib2==0.4.0
 + google-genai==2.10.0
 + google-generativeai==0.8.6
 - grpcio-status==1.81.1
 + grpcio-status==1.71.2
 + httplib2==0.32.0
 - langchain-core==0.3.72
 + langchain-core==1.4.8
 - langchain-google-genai==2.1.8
 + langchain-google-genai==4.2.6
 + langchain-protocol==0.0.18
 - protobuf==6.33.6
 + protobuf==5.29.6
 + pyparsing==3.3.2
 + uritemplate==4.2.0


In [25]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

## Data Ingestion


In [26]:
from langchain.document_loaders import TextLoader

In [27]:
loader = TextLoader("/Users/palla/Desktop/LLMOPS/data/Agentic AI.txt", encoding="utf8")
documents = loader.load()

In [28]:
documents[0].page_content[:500]  # Print the first 500 characters of the first documen

'Understanding Agentic AI\n\nAgentic AI refers to a new paradigm in artificial intelligence where systems are designed not just to respond to queries or perform specific tasks, but to operate autonomously towards achieving predefined goals. This involves capabilities such as planning, reasoning, executing actions, and continuously adapting to dynamic environments without constant human intervention.\n\nKey Characteristics of Agentic AI\n\nAgentic AI systems are distinct from traditional AI models due t'

In [29]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [30]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=20)

In [31]:
text_chunks=text_splitter.split_documents(documents)

In [32]:
text_chunks

[Document(metadata={'source': '/Users/palla/Desktop/LLMOPS/data/Agentic AI.txt'}, page_content='Understanding Agentic AI'),
 Document(metadata={'source': '/Users/palla/Desktop/LLMOPS/data/Agentic AI.txt'}, page_content='Agentic AI refers to a new paradigm in artificial intelligence where systems are designed not just to respond to queries or perform specific tasks, but to operate autonomously towards achieving'),
 Document(metadata={'source': '/Users/palla/Desktop/LLMOPS/data/Agentic AI.txt'}, page_content='towards achieving predefined goals. This involves capabilities such as planning, reasoning, executing actions, and continuously adapting to dynamic environments without constant human intervention.'),
 Document(metadata={'source': '/Users/palla/Desktop/LLMOPS/data/Agentic AI.txt'}, page_content='Key Characteristics of Agentic AI'),
 Document(metadata={'source': '/Users/palla/Desktop/LLMOPS/data/Agentic AI.txt'}, page_content='Agentic AI systems are distinct from traditional AI model

In [33]:
! uv pip install faiss-cpu


Using Python 3.12.13 environment at: c:\Users\palla\Desktop\LLMOPS\.venv
Checked 1 package in 51ms


In [36]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain.vectorstores import FAISS

In [72]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
)

In [73]:
vectorstore=FAISS.from_documents(text_chunks, embeddings)

In [53]:
vectorstore

In [54]:
retriever=vectorstore.as_retriever()

In [55]:
# Perform similarity search
query = "What is the Key Characteristics of Agentic AI?"
docs = vectorstore.similarity_search(query, k=4)

# Display the results
for i, doc in enumerate(docs):
    print(f"Document {i+1}:")
    print(doc.page_content)
    print("-" * 50)


Document 1:
Key Characteristics of Agentic AI
--------------------------------------------------
Document 2:
Understanding Agentic AI
--------------------------------------------------
Document 3:
The Future of Agentic AI
--------------------------------------------------
Document 4:
Applications of Agentic AI
--------------------------------------------------


In [56]:
from langchain.prompts import ChatPromptTemplate

template="""You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""

In [57]:
prompt=ChatPromptTemplate.from_template(template)

In [58]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse ten sentences maximum and keep the answer concise.\nQuestion: {question}\nContext: {context}\nAnswer:\n"), additional_kwargs={})])

In [59]:
from langchain.schema.output_parser import StrOutputParser

In [60]:
output_parser=StrOutputParser()

In [67]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm_model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    api_key=os.getenv("GOOGLE_API_KEY"),
)

In [68]:
from langchain.schema.runnable import RunnablePassthrough


rag_chain = (
    {"context": retriever,  "question": RunnablePassthrough()}
    | prompt
    | llm_model
    | output_parser
)

In [69]:
rag_chain.invoke("tell me about Agentic AI")

'Agentic AI represents a new paradigm in artificial intelligence. In this approach, systems are designed to operate autonomously towards achieving a goal. This differs from traditional AI, which typically focuses on responding to queries or performing specific tasks.'